In [ ]:
#---------------Essential libraries------------------ 
import pandas as pd  
import numpy as np
#--------------visulation libraries----------------
import matplotlib.pyplot as plt 
%matplotlib inline 
import seaborn as sns
#-------------sklean essentials-----------
from sklearn.pipeline import Pipeline
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV

## **step 1. load dataset**
---
Goal : Predict California house prices using housing features such as income, average rooms, population, and location. 


In [ ]:
housing= fetch_california_housing(as_frame=True)

df=pd.DataFrame(data=housing.data,columns=housing.feature_names)
print(f'dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
df['Price']=housing.target # adding target variable to df 
df.head()

## **step 2. Explore and understand the dataset** 
**we inspect:**
 -Data types 
 -Missing values 
 -Statistical summery 
 -Duplicate record

In [ ]:
print("==Data types & non-null counts===")
df.info()

In [ ]:
print(df.isnull().sum())
print('=='*30)
print(f"total number of missing values: {df.isnull().sum().sum()}")
print('--'*30)

In [ ]:
print('-'*30)
print("== Statistical report==")
print('-'*30)
df.describe()


In [ ]:
print(df.duplicated().sum() )#missing duplicate check

## **step 3. Feature engineering :**

In [ ]:
#can i calculate cost of one room ? no because of dataleakage (as price is target variable)
# we don't know the price before the model predicts 
df["Rooms_Per_Household"]=df['AveRooms']/df['AveOccup']
df['BedRooms_Per_Rooms']=df['AveBedrms']/df['AveRooms']
df["population_per_household"]=df["Population"]/df["AveOccup"]
print(' new feature created :      |')
print("--"*20)
print(' 1.-> Rooms per household : |')
print(' 2.-> Bed Rooms per Room  : |')
print(' 3.-> poputaion per house : |')

## **step 4. visulise the dataset**

In [ ]:
df.hist(figsize=(15,10),bins=30)
plt.tight_layout()
plt.show()



In [ ]:
sns.scatterplot(
    x='MedInc',
    y='Price',
    data=df,
    )
plt.tight_layout()
plt.show()


In [ ]:
#box plot show outliers :
plt.figure(figsize=(8,5))
sns.boxplot(data=df)
plt.xticks(rotation=45)
plt.title('Box plot')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['Price'], kde=True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['MedInc'], kde=True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['Population'], kde=True)

plt.tight_layout()
plt.show()

*Here, we can clearly see population has do many outliers*

In [ ]:
# Pair plot
sns.pairplot(df.sample(2000 , random_state=42))
plt.tight_layout()
plt.show()

In [ ]:
# ploting heatmap to see correlations 
plt.figure(figsize=(10,10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('correlation matrix of features')
plt.tight_layout()
plt.show()

## **step 5. Build & train ML models**

In [ ]:
#independent features and dependent features 
x=df.drop(columns=['Price']) ## all features 
y=df['Price']# all targets(outputs)

print(f'features (x): {x.shape[1]} columns')
print(f'target (y):{y.shape}')

In [ ]:
# train test my data 
# we will use 70% of data to train and 30% to test.

x_train, x_test, y_train, y_test=train_test_split(x,y,test_size=0.20,random_state=42)

In [ ]:
# standerdizing the dataset
# bring all the features to same scale 
scaler=StandardScaler()# initialisiation
x_train_sc=scaler.fit_transform(x_train)
x_test_sc=scaler.transform(x_test)

print(f'Training set size : {x_train.shape[0]:,} samples')
print(f' Test    set size : {x_test.shape[0]:,} sample')


In [ ]:
#---------model 1 . Linear Regression--------------
# Think of it as drawing a straight line that fits all the data points per
lr=LinearRegression()
lr.fit(x_train_sc,y_train)
print('👍Linear Regression trained!')

#model 2: ridge Regression
rid=Ridge()
rid.fit(x_train_sc,y_train)
print('👍ridge regression trained!')

rf_model=RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(x_train, y_train) #tree based model don't require scaled inputs 
print('👍RandomForest Regression trained!')



## **step 6.Evaluate the models**

In [ ]:
def evaluate_model(name , model, x_test_data,y_test_data):
    y_pred = model.predict(x_test_data)
    
    mae=mean_absolute_error(y_test_data,y_pred)
    mse=np.sqrt(mean_squared_error(y_test_data,y_pred))
  
    r2=r2_score(y_test_data,y_pred)
    print(f'\n{'='*50}')
    print(f'📌 {name}')
    print(f'{'='*50}')
    print(f'mean absolute error: {mae:.4f}')
    print(f'{'--'*30}')
    print(f'mean squared error: {mse:.4f}')
    print(f'{'--'*30}')
    print(f'R2 score: {r2:.4f}')
    print()
    print(f'{'**'*30}')

lr_report=evaluate_model('linear regression',lr,x_test_sc,y_test)
rid_report=evaluate_model('ridge regression',rid,x_test_sc,y_test)
rf_model_report=evaluate_model('randomforest regression',rf_model,x_test,y_test)


In [ ]:
# regression plt : it draws the best fit line especially use for linear regression
sns.regplot(x="MedInc",y=y,data=df,scatter_kws={"color":"blue"},line_kws={"color":"red"})
plt.show()

## **hyperparameter tuning **

In [ ]:

#1 Define parameter grid
parameter={
    'n_estimators': [50, 100, 200],      
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
}
    

#2. set up RandomizedSearchCV
rf_random=RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=parameter,
    n_iter=10,   # tries 10 random combinations
    cv=3,        # 3-fold cross validation
    verbose=2,
    random_state=42,
    n_jobs=-1    # use all available CPU cores
    )

In [ ]:
print("Tuning hyperparameter......")
rf_random.fit(x_train,y_train)
best_rf_model=rf_random.best_estimator_  #cann't use _best_param_ because it is dictionary of hyperparameters not a fitted model   
                                          # best_estimators_ : already refit on full train data by default   
print("Best parameters found: ",best_rf_model)

In [ ]:
# y_pred_tuned = best_rf.predict(x_test)
evaluate_model("Tunned random forest:",best_rf_model,x_test,y_test)

In [ ]:
ridge_pipe=Pipeline([('scaler',StandardScaler()),("ridge",Ridge())])
parameters={'ridge_alpha':[1,2,5,10,20,30,40,50,60,70,80,90]}
ridgecv=GridSearchCV(rid,parameters,scoring='neg_mean_squared_error',cv=5)
ridgecv.fit(x_train,y_train)
ridge_pred=ridgecv.predict(x_test)

In [ ]:
ridgecv.best_params_

In [ ]:
ridgecv.best_score_

In [ ]:
score=r2_score(y_test,ridge_pred)
score